📝 [최종 모의고사 2차] 작업형 제1유형
문제 1. (시계열 차이 & 그룹핑 - 난이도 상) Machine_ID가 **'M_01'**인 데이터만 추출합니다. 데이터를 시간(Time) 순서대로 정렬한 후, 직전 시간 대비 Temperature(온도)의 **변동폭(절댓값 차이)**을 계산합니다. 이 변동폭이 가장 컸던 시간(Time)의 **'월(Month)'**을 구하시오. (단, 정수로 출력하시오.)

문제 2. (문자열 포함 여부 & 조건부 집계) Status 컬럼에 **'Error'**라는 단어가 포함된 데이터만 필터링합니다. 필터링된 데이터 중에서, Vibration(진동) 컬럼의 결측치를 해당 데이터들의(필터링된 데이터의) 평균값으로 채웁니다. 그 후, 보정된 Vibration 값이 7 이상인 데이터의 개수를 구하시오.

문제 3. (Min-Max Scaling & 시간 조건) Time 컬럼에서 **'오전(6시~11시, 6<=x<=11)'**에 해당하는 데이터만 추출합니다. 이 데이터의 Power_Usage 컬럼을 Min-Max Scaling을 적용하여 변환합니다. (sklearn 미사용 권장) 변환된 값이 0.5 보다 큰(> 0.5) 데이터들의 원래 Power_Usage 평균값을 구하시오. (단, 소수점 이하는 버리고 **정수(int)**로 출력하시오.)

In [16]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n_rows = 3000

# 1. 데이터 생성
times = pd.date_range('2023-01-01', periods=n_rows, freq='h')
machines = np.random.choice(['M_01', 'M_02', 'M_03', 'M_04', 'M_05'], n_rows)

# 상태 코드 생성 (정상, 경고, 에러)
status_opts = ['[Normal]', '[Warning: Overheat]', '[Warning: Noise]', '[Error: 001]', '[Error: 002]']
status_list = np.random.choice(status_opts, n_rows, p=[0.8, 0.1, 0.05, 0.03, 0.02])

df = pd.DataFrame({
    'Time': times,
    'Machine_ID': machines,
    'Temperature': np.random.normal(60, 10, n_rows),
    'Vibration': np.random.normal(5, 2, n_rows),
    'Power_Usage': np.random.randint(100, 500, n_rows),
    'Status': status_list
})

# 결측치 주입 (Vibration)
df.loc[np.random.choice(df.index, 50), 'Vibration'] = np.nan

# 날짜 정렬 (뒤죽박죽 섞음 -> 문제에서 정렬 유도)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("최종 모의고사 2차 데이터 생성 완료!")
print(df.head())

최종 모의고사 2차 데이터 생성 완료!
                 Time Machine_ID  Temperature  Vibration  Power_Usage  \
0 2023-03-17 01:00:00       M_04    47.091690   4.449014          187   
1 2023-02-19 14:00:00       M_02    70.419261   5.595863          221   
2 2023-03-17 17:00:00       M_01    69.790871   5.424813          247   
3 2023-01-11 11:00:00       M_05    53.821372   4.643010          401   
4 2023-04-15 09:00:00       M_04    41.590929   4.636016          300   

             Status  
0          [Normal]  
1          [Normal]  
2  [Warning: Noise]  
3          [Normal]  
4          [Normal]  


In [36]:
# 1번 문제
df['Time'] = pd.to_datetime(df['Time'])
df_1 = df[df['Machine_ID']=='M_01']
df_1 = df_1.sort_values(by='Time', ascending=False)
print(df_1.head())

df_1['Diff'] = df_1['Temperature'].diff().abs()
idx = df_1['Diff'].idxmax()

result = df.loc[idx]
print(result)
# 답: 2월

                    Time Machine_ID  Temperature  Vibration  Power_Usage  \
399  2023-05-05 20:00:00       M_01    81.391611   2.350236          145   
2455 2023-05-05 16:00:00       M_01    67.835137   8.010379          210   
2296 2023-05-05 08:00:00       M_01    68.910092   6.160863          125   
1967 2023-05-04 19:00:00       M_01    64.416013   6.907532          298   
120  2023-05-04 15:00:00       M_01    76.623780   7.351757          256   

                   Status  hours  
399              [Normal]     20  
2455         [Error: 002]     16  
2296             [Normal]      8  
1967             [Normal]     19  
120   [Warning: Overheat]     15  
Time           2023-02-09 01:00:00
Machine_ID                    M_01
Temperature              87.456967
Vibration                 6.612338
Power_Usage                    310
Status         [Warning: Overheat]
hours                            1
Name: 943, dtype: object


In [ ]:
# 2번 문제
df_2 = df[df['Status'].str.contains('Error')].copy()
mean_error = df_2['Vibration'].mean()
df_2['Vibration'] = df_2['Vibration'].fillna(mean_error)

result = len(df_2[df_2['Vibration'] >= 7])
print(result)
# 답: 20

20


In [ ]:
# 3번 문제
df['Time'] = pd.to_datetime(df['Time'])
df['hours'] = df['Time'].dt.hour
q1 = df[(df['hours'] >= 6) & (df['hours'] <= 11)].copy()
#print(q1)

min_p = q1['Power_Usage'].min()
max_p = q1['Power_Usage'].max()

q1['Scaled'] = (q1['Power_Usage'] - min_p) / (max_p - min_p)
mean_s = q1[q1['Scaled'] > 0.5]['Power_Usage'].mean()
print(int(mean_s))
# 답: 397

397


📝 [제2유형] 여행자 보험 가입 예측 (Simple Ver.)
문제: 제공된 학습 데이터(train)를 이용하여 고객의 보험 가입(청구) 여부(Claim)를 예측하는 모델을 만들고, 평가용 데이터(test)에 대한 예측 결과를 result.csv 파일로 제출하시오.

1. 데이터 설명

Target: Claim (1: 가입, 0: 미가입)

Features:

Age: 나이

Agency, Agency Type: 범주형 변수 (인코딩 필요)

Commision, Duration: 수치형 변수 (Duration에 결측치 있음)

2. 제출 형식

result.csv 파일로 저장.

**ID**와 pred 두 개의 컬럼만 포함.

pred 컬럼에는 **가입할 확률(Probability)**을 제출하시오. (0~1 사이 실수)

3. 평가 지표

ROC-AUC Score

In [48]:
import pandas as pd
import numpy as np

# 랜덤 시드 고정
np.random.seed(2025)
n_rows = 1500

# 1. 데이터 생성 (순수 numpy 사용)
data = {
    'ID': range(1001, 1001 + n_rows),
    'Age': np.random.randint(20, 80, n_rows),
    'Agency': np.random.choice(['Agency_A', 'Agency_B', 'Agency_C'], n_rows),
    'Agency Type': np.random.choice(['Airlines', 'Travel Agency'], n_rows),
    'Commision': np.random.uniform(0, 200, n_rows),
    'Duration': np.random.randint(1, 100, n_rows),
    'Claim': 0 # 초기화
}

df = pd.DataFrame(data)

# 2. Target 생성 (나이 많고, 수수료 높을수록 가입 확률 높음)
# 간단한 로직: 점수 > 평균이면 1, 아니면 0 (노이즈 추가)
score = df['Age'] * 0.5 + df['Commision'] * 2 + np.random.normal(0, 20, n_rows)
threshold = score.median()
df['Claim'] = np.where(score > threshold, 1, 0)

# 3. 결측치 주입 (Duration)
df.loc[np.random.choice(df.index, 50), 'Duration'] = np.nan

# 4. Train / Test 분리 (수동 분리)
train = df.iloc[:1000]
test = df.iloc[1000:].drop('Claim', axis=1) # Test에는 타겟 제거

print("데이터 생성 완료!")
print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(train.head())

데이터 생성 완료!
Train shape: (1000, 7), Test shape: (500, 6)
     ID  Age    Agency    Agency Type   Commision  Duration  Claim
0  1001   50  Agency_B  Travel Agency  147.538421      75.0      1
1  1002   38  Agency_B  Travel Agency   57.659979      50.0      0
2  1003   50  Agency_B       Airlines  135.676047      94.0      1
3  1004   32  Agency_A       Airlines  110.596363       NaN      1
4  1005   76  Agency_C  Travel Agency   45.992067      67.0      0


In [52]:
#print(test.info())
# 인코딩: Agency, Agency Type
# 결측치: Duration

mean_d = train['Duration'].mean()
train['Duration'] = train['Duration'].fillna(mean_d)
test['Duration'] = test['Duration'].fillna(mean_d)

X = train.drop(['ID', 'Claim'], axis=1)
y = train['Claim']
X_submit = test.drop(['ID'], axis=1)

cols = ['Agency', 'Agency Type']
ct = pd.concat([X, X_submit])
ct = pd.get_dummies(ct, columns=cols)
X = ct.iloc[:len(X)]
X_submit = ct.iloc[len(X):]

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
pred_proba = model.predict_proba(X_val)[:,1]

from sklearn.metrics import roc_auc_score
score = roc_auc_score(y_val, pred_proba)
print(round(score, 3))

model = RandomForestClassifier(random_state=42)
model.fit(X, y)
pred = model.predict_proba(X_submit)[:,1]

result = pd.DataFrame({
    'ID': test['ID'],
    'pred': pred
})

#result.to_csv('result.csv', index=False)
print(result.head())

C:\Users\sangh\AppData\Local\Temp\ipykernel_13052\2200849869.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Duration'] = train['Duration'].fillna(mean_d)


0.986
        ID  pred
1000  2001  0.97
1001  2002  0.97
1002  2003  0.12
1003  2004  0.00
1004  2005  0.98


📝 [제3유형] 회귀분석 심화
문제 1. (다중선형회귀 - 회귀계수) 직원의 성과 점수(Performance)를 종속변수로 하고, Years_Exp(경력), Salary(연봉), Job_Sat(만족도)를 독립변수로 하는 다중선형회귀(OLS) 모델을 학습시키시오. 이때, **Years_Exp 변수의 회귀계수(Coefficient)**를 구하시오. (단, statsmodels를 사용하며, 결과는 반올림하여 소수 셋째 자리까지 출력하시오.)

문제 2. (로지스틱 회귀 - 오즈비) 직원의 퇴사 여부(Attrition)를 종속변수로 하고, Age, Salary, Job_Sat을 독립변수로 하는 로지스틱 회귀(Logit) 모델을 학습시키시오. 이때, Job_Sat(직무 만족도)가 1단위 증가할 때, 퇴사할 오즈(Odds)는 몇 배가 되는지(오즈비) 구하시오. (단, 결과는 반올림하여 소수 넷째 자리까지 출력하시오.)

문제 3. (상관분석 - 통계량) Salary(연봉)와 Performance(성과 점수) 간의 **피어슨 상관계수(Pearson Correlation Coefficient)**를 구하시오. (단, scipy.stats를 사용하며, 결과는 반올림하여 소수 셋째 자리까지 출력하시오.)

In [53]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n = 400

# 1. 독립변수 생성
data = {
    'ID': range(1001, 1001 + n),
    'Age': np.random.randint(24, 60, n),
    'Years_Exp': np.random.randint(1, 20, n),
    'Job_Sat': np.random.randint(1, 6, n)
}
df = pd.DataFrame(data)

# Salary: 경력과 나이에 비례
df['Salary'] = 30000 + (df['Years_Exp'] * 2000) + (df['Age'] * 500) + np.random.normal(0, 5000, n)

# 2. 종속변수 1 (Performance): 경력과 만족도가 높을수록 높음
# 식: 20 + 2*Exp + 5*Sat + Error
df['Performance'] = 20 + (df['Years_Exp'] * 2.0) + (df['Job_Sat'] * 5.0) + np.random.normal(0, 5, n)
df['Performance'] = df['Performance'].clip(0, 100)

# 3. 종속변수 2 (Attrition): 만족도가 낮고 연봉이 낮을수록 퇴사 확률 높음
# 식: Logit = 2 - 0.8*Sat - 0.00005*Salary
logits = 2 - (0.8 * df['Job_Sat']) - (0.00005 * df['Salary'])
probs = 1 / (1 + np.exp(-logits))
df['Attrition'] = np.random.binomial(1, probs)

print("데이터 생성 완료!")
print(df.head())

데이터 생성 완료!
     ID  Age  Years_Exp  Job_Sat         Salary  Performance  Attrition
0  1001   54         10        1   82074.907623    41.559047          0
1  1002   42         19        3   90736.044002    67.836166          0
2  1003   54         19        1  102151.239802    70.295290          0
3  1004   36         18        3   86337.519682    78.821604          0
4  1005   27          1        4   43705.411846    39.830261          0


In [56]:
from statsmodels.formula.api import ols
model1 = ols('Performance ~ Years_Exp + Salary + Job_Sat', data=df).fit()
coef_1 = model1.params['Years_Exp']
print(round(coef_1, 3))

from statsmodels.formula.api import logit
import numpy as np
model2 = logit('Attrition ~ Age + Salary + Job_Sat', data=df).fit()
odds_ratio = np.exp(model2.params['Job_Sat'])
print(round(odds_ratio, 4))

from scipy.stats import pearsonr
stat, p_val = pearsonr(df['Salary'], df['Performance'])
print(round(stat, 3))

1.972
Optimization terminated successfully.
         Current function value: 0.088568
         Iterations 9
0.4478
0.67
